# The top agent is a 720-turn replay, not a strategy

I spent a day trying to make my Kaggriculture agent better and got nowhere, because
I was measuring the wrong thing. Two findings, both cheap to check, both of which
changed what I built:

1. **Money against the built-in `starter` opponent barely moves.** The rebuild below
   is worth +167% win rate against real agents and only **+15%** money against
   `starter` ($68,052 to $78,692, ten seeds). `starter` never touches the market, so
   the price holes the town eats stay open all season and every farm looks rich:
   the same agent banks $68,052 against it and $29,444 against a leaderboard agent.
   Tune against `starter` and almost nothing you learn transfers.
2. **The strongest public agent is not a policy at all.** It is a hardcoded list of
   720 actions — one per turn for the whole season — with a small reactive layer to
   repair weeds and re-time harvests. Whatever the top of this leaderboard is doing,
   it is offline sequence optimisation, not a cleverer heuristic.

This notebook shows how to check both, publishes the harness that does it, and
measures exactly where the money gap comes from. My agent is in here too, rebuilt
on what the measurement said; it went from **3.0% to 8.0%** win rate against the
top five public agents. That is still losing. I would rather publish the honest
number and the method than a number I cannot reproduce.

**Credit.** The replay I take apart is from
[boatlee's V16-RC5](https://www.kaggle.com/code/boatlee/v16-rc5-high-score-8c-4s-premium-market-lead).
The other four sparring partners are
[kaitofukami](https://www.kaggle.com/code/kaitofukami/25-27-strict-future-v27-midgame-meta-reset),
[pilkwang](https://www.kaggle.com/code/pilkwang/kaggriculture-structured-economic-policy),
[romantamrazov](https://www.kaggle.com/code/romantamrazov/kaggriculture-hamburger) and
[salemali7](https://www.kaggle.com/code/salemali7/3094-score-kaggriculture).
I do not redistribute anyone's agent source here — the extractor below pulls it from
their published notebooks on your own machine, and the numbers I quote are
measurements of that code, not the code.

The market analysis this agent prices from is my earlier
[Strawberry pays 24x what the price table says](https://www.kaggle.com/code/nekkon/strawberry-pays-24x-what-the-price-table-says),
corrected here for 1.32.7 — thanks to **destbreso** for catching that the published
table was computed on 1.32.6.

In [ ]:
import json, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "kaggle-environments>=1.32.7"], check=True)
import kaggle_environments
from kaggle_environments.envs.kaggriculture.kaggriculture import (
    MARKET_PARAMS, HINGE_GAIN, LAND_PRICES, FARM_HAND_COST_MULT, market_price,
)
print("kaggle-environments", kaggle_environments.__version__)
print("land prices", LAND_PRICES, "| hire cost multiplier", FARM_HAND_COST_MULT)

The agent itself and the measurements are loaded first, so every demo below runs.
`main.py` is discussed in section 7; the JSON holds the arena results, which were
produced offline against agents you extract yourself (section 2).

In [ ]:
%%writefile main.py
"""Kaggriculture: sell into the hole, not into the market.

The whole game turns on a sign. Market inventory starts at I0 = 10,000 and moves
both ways: players selling push it up, the town eating pulls it down. Price is a
function of the *distance* from I0, and for most products the lower branch is far
steeper than the upper one. So what a product is worth is set not by its base
price but by how much the town has already eaten out of it, and by the shape of
that lower curve.

Computed from the engine's own market_price, accounting for shops unlocking
gradually (days 3, 6, 9, ... up to 8 instances, drawn with replacement):

    product       town eats     fill the hole     dump into a flat market
    STRAWBERRY        426          $100,445           $4,173     (x24)
    MILK              327          $ 86,662           $6,432     (x13.5)
    WOOL              228          $ 54,340
    WHEAT             525          $ 21,152
    TOMATO            228          $ 16,812
    CARROT            327          $ 13,246
    EGG               228          $ 12,972
    MELON              30          $  8,184

Per tile per season, after seed or livestock cost:

    cow $8,333    sheep $7,343    melon $4,092    goose $2,820
    strawberry $1,674    wheat $1,209    tomato $737

Hence the layout: pastures with cows and sheep first, then melon, then geese,
with wheat pulled in behind them as feed. Strawberry is the most valuable thing
to *sell* and a mediocre thing to *grow* -- a plant yields four units in
seventeen days.

The trading rule: sell only while the price is above base, because once
inventory reaches I0 the hole is full and the next unit goes up the shallow
branch, where the same strawberry is worth a tenth as much. The agent
re-implements market_price to work out exactly how many units it can sell before
crossing that threshold, and holds the rest. Two corrections make it survive
contact with an opponent:

  * the threshold decays through the season, because the hole is shared and
    holding out for a price that never comes leaves you with a full shed;
  * liquidity beats price -- below two days of running costs it sells whatever
    it has at whatever it fetches.

On the last days it dumps everything: unsold goods do not count.
"""


P = {
    # Target layout, from allocating tiles greedily by marginal revenue: while
    # the next cow tile pays more than the next sheep tile, take the cow. The
    # order comes out cows -> sheep -> melon -> geese, with wheat pulled in as
    # feed. These caps were then trimmed by the sweep: fewer head than the
    # static optimum, because servicing an animal costs four actions a day.
    "cows": 8,
    "sheep": 4,
    "melons": 20,
    "geese": 0,
    "straw": 36,          # fertilised strawberry yields 8 units off one tile
    "fert_straw": 1,      # spend collected fertilizer on strawberry, not sell it
    "wheat_per_animal": 0.5,
    "wheat_fill": 40,     # wheat is the filler crop on every otherwise idle tile
    "buy_animal_until": 22,     # later than this a cow never pays back: first milk on day 8
    "plant_until": {"MELON": 16, "WHEAT": 26, "STRAWBERRY": 14},
    # --- trading
    # The hole is shared with the opponent: whoever sells first takes the
    # premium. So the threshold decays through the season -- holding is only
    # worth it while there are days left for the town to eat more.
    "floor_start": 1.15,   # early on, sell only above base
    "floor_end": 0.8,      # by the end, at almost any price
    # Holding stock is a luxury for the solvent. The market is shared: if the
    # opponent is filling the hole, the price never recovers, and a farm with
    # no cash does nothing. Below this many days of running costs, sell
    # everything at whatever it fetches.
    "cash_floor_days": 2,
    "dump_day": 28,        # from this day, dump everything: leftovers score nothing
    "wheat_days": 2,       # days of feed kept in the shed
    # --- labour and money
    "hands": 10,
    "hire_max": 250,
    "carry": 10,
    "shed_cap": 55,
    "hold_until": 30,     # below this shed load, holding is free
    "drop_load": 6,
    "evening": 17,
    "hunger": 4,
    "runway": 3,
    "hire_day_cost": 200,
    "land_day_min": 1,
    "land_free_left": 100,
    "land_stop": 2,      # quadrants to buy: 2 == NE and SW, skipping the $4k SE
}

CROPS = {
    "WHEAT":      {"seed": 10, "first": 2, "max_day": 4,  "ongoing": False},
    "CARROT":     {"seed": 20, "first": 2, "max_day": 3,  "ongoing": False},
    "TOMATO":     {"seed": 50, "first": 8, "max_day": 8,  "ongoing": True, "interval": 1, "n": 4},
    "STRAWBERRY": {"seed": 100, "first": 10, "max_day": 10, "ongoing": True, "interval": 2, "n": 4},
    "MELON":      {"seed": 80, "first": 10, "max_day": 12, "ongoing": False},
}
ANIMALS = {
    "GOOSE": {"cost": 300, "struct": "COOP",    "product": "EGG"},
    "COW":   {"cost": 400, "struct": "PASTURE", "product": "MILK"},
    "SHEEP": {"cost": 500, "struct": "PASTURE", "product": "WOOL"},
}
# from the engine's MARKET_PARAMS; needed to work out how much can be sold
MP = {
    "WHEAT":      (25, 400, "sqrt", 0.80, "log", 0.20),
    "CARROT":     (35, 450, "hinge", 1.00, "sqrt", 0.70),
    "TOMATO":     (60, 200, "hinge", 0.40, "sqrt", 0.60),
    "STRAWBERRY": (120, 100, "sqrt", 0.70, "linear", 1.60),
    "MELON":      (250, 300, "log", 0.20, "sq", 3.60),
    "EGG":        (50, 332, "hinge", 0.40, "log", 0.20),
    "MILK":       (160, 122, "sqrt", 0.60, "linear", 1.60),
    "WOOL":       (200, 105, "log", 0.20, "sq", 3.20),
    "FERTILIZER": (100, 200, "linear", 0.40, "linear", 0.40),
}
HINGE_GAIN = 8.0
I0 = 10000
PRODUCTS = list(MP)
SELLABLE = ["MILK", "WOOL", "STRAWBERRY", "MELON", "EGG", "TOMATO",
            "CARROT", "FERTILIZER", "WHEAT"]


def _shape(f, x, T=None):
    import math
    x = max(0.0, x)
    if f == "linear":
        return x
    if f == "sq":
        return x * x
    if f == "sqrt":
        return math.sqrt(x)
    if f == "log":
        return math.log(1.0 + x)
    if f == "hinge":
        # Linear in x/T below the knee at T; the quadratic only fires past it.
        if not T or T <= 0:
            return x
        u = x / T
        return u + HINGE_GAIN * max(0.0, u - 1.0) ** 2
    return x


def price_of(item, inv):
    base, T, bf, bt, af, at = MP[item]
    if inv < I0:
        amp = bt * base / _shape(bf, T, T)
        p = base + amp * _shape(bf, I0 - inv, T)
    else:
        amp = at * base / _shape(af, T, T)
        p = base - amp * _shape(af, inv - I0, T)
    return max(1, int(round(p)))


def sellable_units(item, inv, floor_price):
    """How many units go before the price drops below floor_price.

    Each sale raises inventory by one, so the price falls as the order fills.
    This walks the curve exactly the way the engine does.
    """
    n = 0
    while n < 200:
        if price_of(item, inv + n) < floor_price:
            break
        n += 1
    return n


def _get(d, key, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return getattr(d, key, default)


def _shed_tiles(n):
    h = n // 2
    return [(h - 1, h - 1), (h, h - 1), (h - 1, h), (h, h)]


def _step(fx, fy, tx, ty):
    if fx < tx:
        return "EAST"
    if fx > tx:
        return "WEST"
    if fy < ty:
        return "SOUTH"
    if fy > ty:
        return "NORTH"
    return None


def _d(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def agent(obs):
    player = _get(obs, "player", 0)
    farms = _get(obs, "farms", [])
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    private = _get(obs, "private", {}) or {}
    shed = dict(_get(private, "shed", {}) or {})
    seeds = dict(_get(private, "seeds", {}) or {})
    invs = [dict(i) if i else {} for i in (_get(private, "inventories", [{}]) or [{}])]
    tiles = farm["tiles"]
    n = len(tiles)
    day, hour = _get(obs, "day", 0), _get(obs, "hour", 0)
    money = farm["money"]
    mkt = _get(obs, "market", {}) or {}
    inv_mkt = _get(mkt, "inventory", {}) or {}
    prices = _get(mkt, "prices", {}) or {}
    wheat_price = max(1, prices.get("WHEAT", 25))

    units = [tuple(farm["farmer"])] + [tuple(p) for p in farm.get("hands", [])]
    n_units = len(units)
    while len(invs) < n_units:
        invs.append({})
    shed_access = _shed_tiles(n)
    centre = shed_access[0]

    # -------------------------------------------------------- read the farm
    animals, structs, free, weeds, plants = [], [], [], [], []
    for y in range(n):
        for x in range(n):
            t = tiles[y][x]
            if t is None:
                free.append((x, y))
            elif t == "LOCKED":
                continue
            elif isinstance(t, dict):
                k = t.get("kind")
                if k == "WEED":
                    weeds.append((x, y))
                elif k == "PLANT":
                    plants.append((x, y, t))
                elif "animal" in t:
                    animals.append((x, y, t))
                else:
                    structs.append((x, y, k))       # empty coop or pasture
    free.sort(key=lambda p: _d(p, centre))
    structs.sort(key=lambda s: _d((s[0], s[1]), centre))

    have = {"COW": 0, "SHEEP": 0, "GOOSE": 0}
    unfed = []
    for (x, y, t) in animals:
        have[t["animal"]] += 1
        if not t.get("fed_today"):
            unfed.append((x, y))
    for a in ANIMALS:
        have[a] += shed.get(a, 0) + sum(i.get(a, 0) for i in invs)

    crop_count = {}
    for (x, y, t) in plants:
        crop_count[t["crop"]] = crop_count.get(t["crop"], 0) + 1

    n_animals = sum(have.values())
    wheat_carried = sum(i.get("WHEAT", 0) for i in invs)
    wheat_have = shed.get("WHEAT", 0) + wheat_carried
    shed_total = sum(shed.values())
    day_cost = P["hire_day_cost"] + n_animals * wheat_price
    reserve = P["runway"] * day_cost

    # ------------------------------------------------------------ the market
    # Only 10 orders per turn are accepted and the rest are silently dropped,
    # so the queue is built by priority rather than in the order the code
    # happens to be written. Feed goes first: a shed full of unsold milk once
    # blocked a wheat purchase and the whole herd walked off in two days.
    market = []
    dumping = day >= P["dump_day"]

    # feed: one wheat per head per day; two missed days and it is gone for good
    wheat_target = max(2, int(n_animals * P["wheat_days"]))
    if wheat_have < wheat_target and shed_total < 95 and money > wheat_price:
        want = min(wheat_target - wheat_have, 95 - shed_total,
                   int(money // wheat_price))
        if want > 0:
            market.append(["BUY_PRODUCT", "WHEAT", want])

    # Sell into the hole: below I0 every unit fetches a premium, above it the
    # same strawberry is worth a tenth as much. But holding only works while
    # there is somewhere to put things -- the shed holds 100 and whatever does
    # not fit at nightfall is simply discarded. So the fuller the shed, the
    # lower the price we are willing to accept.
    t = min(1.0, day / max(1, P["dump_day"]))
    floor_day = P["floor_start"] + (P["floor_end"] - P["floor_start"]) * t
    if money < P["cash_floor_days"] * day_cost:
        floor_day = 0.0        # liquidity beats price
    slack = max(0.0, min(1.0, (shed_total - P["hold_until"]) /
                         max(1, 95 - P["hold_until"])))
    floor_mult = floor_day * (1.0 - slack)
    order = sorted((i for i in SELLABLE if shed.get(i, 0) > 0),
                   key=lambda i: -prices.get(i, 0) * shed.get(i, 0))
    for item in order:
        n_have = shed.get(item, 0)
        if item == "WHEAT":
            n_have -= wheat_target
            if n_have <= 0:
                continue
        if dumping:
            market.append(["SELL", item, n_have])
            continue
        base = MP[item][0]
        allow = sellable_units(item, inv_mkt.get(item, I0), base * floor_mult)
        k = min(n_have, allow)
        if k > 0:
            market.append(["SELL", item, k])

    # Hands: the fib(n) price resets every morning, so ten cost $143. Hired
    # across two turns so the hires do not eat the whole order queue at once.
    if hour in (0, 1) and len(market) < 9:
        a, b = 1, 1
        for _ in range(farm.get("hires_today", 0)):
            a, b = b, a + b
        budget = money - n_animals * wheat_price * 2
        while len(market) < 9 and farm.get("hires_today", 0) + \
                sum(1 for o in market if o[0] == "HIRE") < P["hands"]:
            if a > P["hire_max"] or a > budget:
                break
            market.append(["HIRE"])
            budget -= a
            a, b = b, a + b

    # Feed wheat comes before any investment. A herd you cannot feed walks off
    # in two days, and that is the only irreversible loss in the game.
    # Wheat is not only feed. At $10 a seed for 4-6 units into a 525-unit hole
    # it is the cheapest way to turn an idle tile into cash, so it fills every
    # tile the herd and the fruit are not using.
    wheat_want = max(int(round(n_animals * P["wheat_per_animal"])) + 2,
                     min(len(free) + crop_count.get("WHEAT", 0), P["wheat_fill"]))
    wheat_planted = crop_count.get("WHEAT", 0) + seeds.get("WHEAT", 0)
    if wheat_planted < wheat_want and day <= P["plant_until"]["WHEAT"] and free:
        k = min(wheat_want - wheat_planted, len(free), 10,
                int((money - reserve) // CROPS["WHEAT"]["seed"]))
        if k > 0:
            market.append(["BUY_SEED", "WHEAT", k])
            money -= k * 10

    # Animals pay best per tile and per dollar; bought strictly in marginal-
    # revenue order, and only once there is feed growing for them.
    room = len(free) + len(structs)
    # the herd never grows faster than the feed under it
    feed_ok = wheat_planted >= n_animals * P["wheat_per_animal"]
    if room > 0 and shed_total < 90 and day <= P["buy_animal_until"]:
        for kind, cap in (("COW", P["cows"]), ("SHEEP", P["sheep"]),
                          ("GOOSE", P["geese"])):
            if have[kind] >= cap or (kind == "GOOSE" and not feed_ok):
                continue
            cost = ANIMALS[kind]["cost"]
            afford = int((money - reserve) // (cost + P["runway"] * wheat_price))
            k = max(0, min(afford, cap - have[kind], room, 4))
            if k > 0:
                market.append(["BUY_ANIMAL", kind, k])
                money -= k * cost
                room -= k
                break

    # Strawberry: four scheduled yields, doubled to eight if the plant is both
    # watered and fertilised that day, and the town eats 426 of them a season --
    # the deepest hole in the game. This is the crop the livestock farm was
    # crowding out.
    straw_planted = crop_count.get("STRAWBERRY", 0) + seeds.get("STRAWBERRY", 0)
    if (straw_planted < P["straw"] and day <= P["plant_until"]["STRAWBERRY"]
            and len(free) > 1 and money - reserve > 300):
        k = min(P["straw"] - straw_planted, len(free) - 1, 6,
                int((money - reserve) // CROPS["STRAWBERRY"]["seed"]))
        if k > 0:
            market.append(["BUY_SEED", "STRAWBERRY", k])
            money -= k * 100

    # Melon: its hole is only 30 units, but the glut curve starts from $250,
    # so the first hundred still sell dear. A handful of tiles is right.
    melon_planted = crop_count.get("MELON", 0) + seeds.get("MELON", 0)
    if (melon_planted < P["melons"] and day <= P["plant_until"]["MELON"]
            and len(free) > 2 and money - reserve > 400):
        k = min(P["melons"] - melon_planted, len(free) - 2, 4,
                int((money - reserve) // CROPS["MELON"]["seed"]))
        if k > 0:
            market.append(["BUY_SEED", "MELON", k])
            money -= k * 80

    # Land: the 25 NW tiles hold neither the herd nor the feed for it.
    n_extra = len(farm.get("unlocked_quadrants", ["NW"])) - 1
    land_price = ([1000, 2000, 4000][n_extra]
                  if n_extra < min(3, P["land_stop"]) else None)
    if (land_price and day >= P["land_day_min"] and day <= 22
            and len(free) <= P["land_free_left"]
            and money - land_price >= reserve + 600):
        market.append(["BUY_LAND"])

    market = market[:10]

    # ------------------------------------------------------------- the tasks
    # Every job at an animal is done from its own square: arrive, feed, care,
    # harvest, collect. So tasks are grouped by tile, and a unit already
    # standing on one finishes it before moving. Without this rule 83% of all
    # unit-turns went into walking, because units retargeted every turn.
    pending = {}

    def add(p, op):
        pending.setdefault(p, []).append(op)

    for (x, y, t) in animals:
        p = (x, y)
        if not t.get("fed_today"):
            add(p, ["FEED"])
        if t.get("yield_units", 0) > 0:
            add(p, ["HARVEST"])
        if not t.get("cared_today"):
            add(p, ["CARE"])          # +1 on the next scheduled yield
        if t.get("fertilizer_available"):
            add(p, ["COLLECT_FERTILIZER"])
    for (x, y, t) in plants:
        p = (x, y)
        cd = CROPS[t["crop"]]
        age = day - t["planted_day"]
        if not t.get("watered_today"):
            add(p, ["WATER"])
        # Fertiliser doubles an ongoing crop's scheduled yield on days the plant
        # is also watered, and it lasts three days -- so it is worth far more
        # spent on a strawberry than sold at the glut price.
        if (P["fert_straw"] and t["crop"] == "STRAWBERRY"
                and t.get("fertilized_until_day", -1) < day
                and age >= cd["first"] - 2):
            add(p, ["FERTILIZE"])
        if t.get("yield_units", 0) > 0:
            ripe = age >= cd["max_day"] if not cd["ongoing"] else age >= cd["first"]
            if ripe:
                add(p, ["HARVEST"])
    for (x, y) in weeds:
        add((x, y), ["DIG"])

    # what to build on the free tiles
    need_struct = {"PASTURE": max(0, shed.get("COW", 0) + shed.get("SHEEP", 0)
                                  + sum(i.get("COW", 0) + i.get("SHEEP", 0) for i in invs)
                                  - sum(1 for s in structs if s[2] == "PASTURE")),
                   "COOP": max(0, shed.get("GOOSE", 0)
                               + sum(i.get("GOOSE", 0) for i in invs)
                               - sum(1 for s in structs if s[2] == "COOP"))}
    plantable = []
    bi = 0
    for (x, y) in free:
        if need_struct["PASTURE"] > 0:
            add((x, y), ["BUILD_PASTURE"])
            need_struct["PASTURE"] -= 1
        elif need_struct["COOP"] > 0:
            add((x, y), ["BUILD_COOP"])
            need_struct["COOP"] -= 1
        else:
            plantable.append((x, y))
        bi += 1

    # --------------------------------------------------- assigning the units
    actions = [None] * n_units
    order = sorted(range(n_units), key=lambda i: _d(units[i], centre))
    unfed_left = set(unfed)
    targeted = set()
    seeds_left = dict(seeds)
    struct_free = {"COOP": [(x, y) for (x, y, k) in structs if k == "COOP"],
                   "PASTURE": [(x, y) for (x, y, k) in structs if k == "PASTURE"]}
    plant_left = list(plantable)
    wheat_shed = shed.get("WHEAT", 0)
    wheat_en_route = wheat_carried
    animals_shed = {a: shed.get(a, 0) for a in ANIMALS}

    def crop_for_tile():
        """What to plant on the next free tile."""
        for crop in ("STRAWBERRY", "MELON", "WHEAT"):
            if seeds_left.get(crop, 0) > 0:
                return crop
        return None

    for idx in order:
        pos, inv = units[idx], invs[idx]
        has_wheat = inv.get("WHEAT", 0) > 0
        carried_animal = next((a for a in ANIMALS if inv.get(a, 0) > 0), None)
        load = sum(v for k, v in inv.items()
                   if k not in ("WHEAT",) and k not in ANIMALS)
        at_shed = pos in shed_access

        # 1. finish the tile we are standing on
        here = pending.get(pos)
        if here:
            op = None
            for cand in here:
                if cand[0] == "FERTILIZE" and inv.get("FERTILIZER", 0) <= 0:
                    continue          # FERTILIZE spends from this unit's own bag
                if cand[0] == "FEED" and not has_wheat:
                    continue
                op = cand
                break
            if op is not None:
                here.remove(op)
                if op[0] == "FEED":
                    inv["WHEAT"] = inv.get("WHEAT", 0) - 1
                    unfed_left.discard(pos)
                actions[idx] = list(op)
                continue
        if carried_animal:
            st = ANIMALS[carried_animal]["struct"]
            if pos in struct_free[st]:
                struct_free[st].remove(pos)
                inv[carried_animal] = inv.get(carried_animal, 0) - 1
                actions[idx] = ["PLACE", carried_animal]
                continue
        if pos in plant_left:
            crop = crop_for_tile()
            if crop:
                plant_left.remove(pos)
                seeds_left[crop] -= 1
                crop_count[crop] = crop_count.get(crop, 0) + 1
                actions[idx] = ["PLANT", crop]
                continue

        # 2. a trip to the shed
        want = None
        pending_animal = next((a for a, k in animals_shed.items() if k > 0
                               and struct_free[ANIMALS[a]["struct"]]), None)
        if unfed_left and not has_wheat and wheat_shed > 0:
            want = "WHEAT"
        elif pending_animal and not carried_animal and not has_wheat:
            want = pending_animal
        elif load >= P["drop_load"] or (hour >= P["evening"] and load > 0):
            want = "DROP"
        if want:
            if at_shed:
                if want == "WHEAT":
                    take = min(P["carry"], wheat_shed)
                    actions[idx] = ["PICKUP", "WHEAT", take]
                    wheat_shed -= take
                    wheat_en_route += take
                elif want == "DROP":
                    actions[idx] = ["DROP"]
                else:
                    actions[idx] = ["PICKUP", want, 1]
                    animals_shed[want] -= 1
            else:
                tgt = min(shed_access, key=lambda p: _d(pos, p))
                actions[idx] = [_step(*pos, *tgt)]
            continue

        # 3. from mid-morning, the unfed outrank everything else
        if has_wheat and unfed_left and hour >= P["hunger"]:
            tgt = min(unfed_left, key=lambda p: _d(pos, p))
            unfed_left.discard(tgt)
            mv = _step(*pos, *tgt)
            actions[idx] = [mv] if mv else ["FEED"]
            continue

        # 4. head for the densest nearby tile
        best, best_key = None, None
        for p, ops in pending.items():
            if not ops or p in targeted:
                continue
            if all(o[0] == "FEED" for o in ops) and not has_wheat:
                continue
            d = _d(pos, p)
            key = (-len(ops) / (d + 1), d)
            if best_key is None or key < best_key:
                best, best_key = p, key
        if best is None and carried_animal:
            st = struct_free[ANIMALS[carried_animal]["struct"]]
            if st:
                best = min(st, key=lambda p: _d(pos, p))
        if best is None and plant_left and crop_for_tile():
            best = min(plant_left, key=lambda p: _d(pos, p))
        if best is None:
            if load:
                tgt = min(shed_access, key=lambda p: _d(pos, p))
                mv = _step(*pos, *tgt)
                actions[idx] = [mv] if mv else ["DROP"]
            else:
                actions[idx] = ["PASS"]
            continue
        targeted.add(best)
        mv = _step(*pos, *best)
        actions[idx] = [mv] if mv else ["PASS"]

    for i in range(n_units):
        if not actions[i] or actions[i][0] is None:
            actions[i] = ["PASS"]

    return {"farmer": actions[0], "hands": actions[1:], "market": market}

In [ ]:
MEASURED = json.loads(r'''{"seeds": [1000, 1007, 1014, 1021, 1028, 1035, 1042, 1049, 1056, 1063], "replay": {"turns": 720, "hires_per_day": [5, 1, 2, 3, 4, 3, 4, 7, 6, 7, 14, 10, 10, 8, 9, 9, 13, 9, 11, 13, 14, 12, 10, 14, 11, 12, 12, 11, 10, 10], "total_hires": 264, "land_days": [6, 10], "buys": {"WHEAT": 333, "COW": 8, "SHEEP": 4, "MELON": 20, "STRAWBERRY": 37}, "sells": {"WHEAT": 479, "FERTILIZER": 300, "WOOL": 154, "MILK": 320, "MELON": 126, "STRAWBERRY": 300}}, "opmix_before": {"ours": {"money": 28939.0, "moves": 4041, "pass": 1082, "water": 188, "harvest": 198, "plant": 45, "fertilize": 0, "care": 400, "feed": 404, "collect_fert": 383, "pickup": 496, "sells": {"FERTILIZER": 371, "WHEAT": 46, "MILK": 232, "MELON": 18, "WOOL": 94, "EGG": 160}, "buys": {"WHEAT": 47, "COW": 8, "MELON": 21, "SHEEP": 10, "GOOSE": 13}, "unit_turns": 7522}, "boatlee": {"money": 132434.0, "moves": 2855, "pass": 301, "water": 1010, "harvest": 390, "plant": 199, "fertilize": 72, "care": 967, "feed": 290, "collect_fert": 296, "pickup": 135, "sells": {"WHEAT": 479, "FERTILIZER": 300, "WOOL": 154, "MILK": 320, "MELON": 126, "STRAWBERRY": 300}, "buys": {"COW": 8, "SHEEP": 4, "MELON": 20, "WHEAT": 144, "STRAWBERRY": 37}, "unit_turns": 6650}}, "opmix_after": {"ours": {"money": 15466.0, "moves": 4727, "pass": 609, "water": 548, "harvest": 177, "plant": 87, "fertilize": 41, "care": 289, "feed": 287, "collect_fert": 277, "pickup": 222, "sells": {"FERTILIZER": 233, "WHEAT": 74, "MILK": 248, "STRAWBERRY": 100, "WOOL": 78}, "buys": {"WHEAT": 71, "COW": 8, "STRAWBERRY": 38, "SHEEP": 4, "MELON": 20}, "unit_turns": 7480}, "boatlee": {"money": 97547.0, "moves": 2855, "pass": 301, "water": 1010, "harvest": 390, "plant": 199, "fertilize": 72, "care": 967, "feed": 290, "collect_fert": 296, "pickup": 135, "sells": {"WHEAT": 479, "FERTILIZER": 300, "WOOL": 154, "MILK": 320, "MELON": 126, "STRAWBERRY": 300}, "buys": {"COW": 8, "SHEEP": 4, "MELON": 20, "WHEAT": 144, "STRAWBERRY": 37}, "unit_turns": 6650}}, "before": {"label": "before", "per_opponent": {"boatlee": {"games": 20, "winrate": 0.0, "our_money": 29444.15, "their_money": 118178.9}, "kaito": {"games": 20, "winrate": 0.0, "our_money": 25804.75, "their_money": 115656.6}, "pilkwang": {"games": 20, "winrate": 0.0, "our_money": 26139.65, "their_money": 97777.15}, "hamburger": {"games": 20, "winrate": 0.15, "our_money": 42133.8, "their_money": 72113.8}, "salemali": {"games": 20, "winrate": 0.0, "our_money": 32612.9, "their_money": 136883.6}}, "games": 100, "winrate": 0.03, "per_seed_mean": {"1000": 28319.6, "1007": 28738.4, "1014": 32059.4, "1021": 24750.4, "1028": 30554.7, "1035": 27572.7, "1042": 28699.2, "1049": 31912.3, "1056": 57913.4, "1063": 21750.4}}, "after": {"label": "after", "per_opponent": {"boatlee": {"games": 20, "winrate": 0.0, "our_money": 37406.95, "their_money": 101617.45}, "kaito": {"games": 20, "winrate": 0.0, "our_money": 39533.0, "their_money": 103404.05}, "pilkwang": {"games": 20, "winrate": 0.0, "our_money": 43895.4, "their_money": 103523.35}, "hamburger": {"games": 20, "winrate": 0.4, "our_money": 53080.35, "their_money": 64043.2}, "salemali": {"games": 20, "winrate": 0.0, "our_money": 37345.15, "their_money": 116939.7}}, "games": 100, "winrate": 0.08, "per_seed_mean": {"1000": 21994.7, "1007": 22901.3, "1014": 53293.0, "1021": 33294.3, "1028": 46006.2, "1035": 58101.2, "1042": 36198.3, "1049": 60677.5, "1056": 39682.7, "1063": 50372.5}}, "vs_starter": {"before": {"mean": 68052.0, "seed1000": 36022.0}, "after": {"mean": 78692.0, "seed1000": 42028.0}}}''')
print("measured:", ", ".join(MEASURED), "|", MEASURED["after"]["games"], "episodes per configuration")

## 1. Why local money lies

The `starter` agent does almost nothing, so any farm that merely functions buries it.
Worse, it never touches the market, so the price holes the town eats stay open all
season and your produce sells near the top of the curve. Against a real opponent the
same farm sells into a market someone else has already drained.

Two things fall out of that. The absolute money roughly doubles, and — the part that
cost me the day — the *differences between agents* compress to nothing. Both versions
of my agent look fine against `starter`; only one of them is meaningfully less bad
against the leaderboard.

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"seed": 1000}, debug=False)
env.run(["main.py", "starter"])
live, theirs = (s["reward"] for s in env.steps[-1])
print(f"live run, this agent vs starter on seed 1000: ${live:,.0f} to ${theirs:,.0f}\n")

vs_starter, vs_real = MEASURED["vs_starter"], MEASURED
print(f"{'':<18}{'vs starter':>12}{'vs boatlee':>12}{'win% vs the 5':>15}")
for tag in ("before", "after"):
    print(f"{'my agent, ' + tag:<18}"
          f"${vs_starter[tag]['mean']:>11,.0f}"
          f"${vs_real[tag]['per_opponent']['boatlee']['our_money']:>11,.0f}"
          f"{vs_real[tag]['winrate']:>14.1%}")
print("\nMoney against starter moved +15%. Win rate against real agents moved +167%.")

## 2. Getting real opponents

Every strong public agent ships its source inside its notebook, but almost none of
them do it in plain sight — they are `%%writefile` cells, custom `%%agentfile`
magics, or base64/base85 blobs compressed with zlib. This extractor handles all four
shapes. Point it at a pulled notebook and it writes `main.py` next to it.

```bash
kaggle kernels pull boatlee/v16-rc5-high-score-8c-4s-premium-market-lead -p opp/boatlee
```

In [ ]:
import base64, gzip, json as _json, re, zlib
from pathlib import Path

def _inflate(blob: str):
    "Try the four encodings public agents use, return the source if it looks like one."
    for decode in (base64.b64decode, base64.b85decode):
        try:
            raw = decode(blob)
        except Exception:
            continue
        for expand in (lambda r: r, zlib.decompress,
                       lambda r: zlib.decompress(r, -15), gzip.decompress):
            try:
                text = expand(raw).decode("utf-8")
            except Exception:
                continue
            if "def agent" in text or "def act" in text:
                return text
    return None

def extract_agent(nb_path):
    "Recover the agent source from a published Kaggriculture notebook."
    cells = _json.load(open(nb_path))["cells"]
    code = [("".join(c["source"])) for c in cells if c["cell_type"] == "code"]

    for src in code:                                    # (a) %%writefile / %%agentfile
        head = src.split("\n", 1)[0]
        if head.startswith(("%%writefile", "%%agentfile")):
            return src.split("\n", 1)[1]

    joined = "\n".join(code)                            # (b) one long quoted blob
    best = None
    for blob in re.findall(r"['\"]([A-Za-z0-9+/=]{2000,})['\"]", joined):
        text = _inflate(blob)
        if text and (best is None or len(text) > len(best)):
            best = text
    if best:
        return best

    for var in ("_PACKED_SOURCE", "_AGENT_B85_PARTS", "AGENT_B64"):   # (c) split parts
        if var not in joined:
            continue
        tail = joined[joined.index(var):]
        stop = min(x for x in (tail.find("))"), tail.find("]"), len(tail)) if x > 0)
        text = _inflate("".join(re.findall(r"['\"]([A-Za-z0-9+/=]*)['\"]", tail[:stop])))
        if text:
            return text
    return None

print(extract_agent.__doc__)

### The arena

The leaderboard rates paired episodes, so the local metric has to be a win rate, and
the board is not symmetric — seat matters — so every pairing is played both ways.

One trap cost me an hour: on macOS `ProcessPoolExecutor` **spawns** workers rather
than forking them, so module-level globals set in `main()` do not reach the workers.
My first sweep reported byte-identical results for every configuration because the
agent path never changed inside the pool. Pass the paths inside the job tuple.

In [ ]:
import itertools, os
from concurrent.futures import ProcessPoolExecutor

def _play(job):
    a_path, b_path, seed = job
    from kaggle_environments import make                 # import inside the worker
    env = make("kaggriculture", configuration={"seed": seed}, debug=False)
    env.run([a_path, b_path])
    a_money, b_money = (s["reward"] for s in env.steps[-1])
    return seed, a_money, b_money

def arena(ours, opponents, seeds=range(1000, 1070, 7)):
    "Win rate of `ours` against each opponent, both seatings, all seeds."
    jobs, tags = [], []
    for name, path in opponents.items():
        for seed in seeds:
            jobs.append((ours, path, seed)); tags.append((name, "home"))
            jobs.append((path, ours, seed)); tags.append((name, "away"))
    table = {name: {"w": 0.0, "n": 0} for name in opponents}
    with ProcessPoolExecutor() as pool:
        for (name, side), (_, a_money, b_money) in zip(tags, pool.map(_play, jobs)):
            us, them = (a_money, b_money) if side == "home" else (b_money, a_money)
            row = table[name]
            row["n"] += 1
            row["w"] += 1.0 if us > them else 0.5 if us == them else 0.0
    return {n: r["w"] / r["n"] for n, r in table.items()}

print("arena(ours='main.py', opponents={'boatlee': 'opp/boatlee/main.py', ...})")

## 3. What is actually inside the top agent

`V16-RC5`'s `main.py` is a small reactive shell wrapped around one compressed JSON
list. Decompressed, it is 720 entries — one per turn of the season — each a full
`{"farmer": ..., "hands": [...], "market": [...]}` action. Turn 0 alone hires five
farm hands, buys a cow, four sheep, five melon seeds, five wheat seeds and fourteen
wheat to eat.

Reading the whole list tells you the shape of a 120k season without guessing.

In [ ]:
R = MEASURED["replay"]

print(f"{R['turns']} hardcoded turns; {R['total_hires']} hires over the season")
print(f"land bought on days {R['land_days']} (NE and SW; the $4,000 SE quadrant never)")
print("\nherd:   ", {k: v for k, v in R["buys"].items() if k in ("COW", "SHEEP", "GOOSE")})
print("planted:", {k: v for k, v in R["buys"].items() if k not in ("COW", "SHEEP", "GOOSE")})
print("\nsold over the season:")
for item, n in sorted(R["sells"].items(), key=lambda kv: -kv[1]):
    print(f"   {item:<12}{n:>5} units")
print("\nhires per day:", R["hires_per_day"])

Three things in that output are worth arguing with, because they contradict what a
marginal-revenue calculation on the price table tells you to do:

- **The herd is tiny.** Eight cows, four sheep, no geese. My own allocator, run on
  revenue per tile, wanted twenty-seven head. It is wrong because it prices tiles and
  the binding constraint is *actions*: a fed, cared-for, harvested, fertiliser-collected
  animal costs four unit-turns a day forever.
- **Labour is rented daily and heavily.** 264 hires. The cost resets every morning and
  follows the Fibonacci sequence, so the tenth hand of a day costs $55 and the whole
  bench of ten costs $143 — nothing against a $100,000 season.
- **The $4,000 quadrant is never bought.** Two expansions, days 6 and 10, then stop.

## 4. Where my season went

Same seed, same opponent, and every unit-turn of both farms classified. This is the
diagnosis that told me what to change — and it corrected me twice.

Read the turn counts, not the money column: this is a single episode on seed 1000,
which happens to be the rebuilt agent's *worst* seed of the ten ($21,995 mean against
$42,252 overall). The composition of a farm's day barely moves between seeds; what it
banks moves enormously, which is the subject of the last figure.

In [ ]:
import pandas as pd

rows = []
for label, blob, seat in (("mine, before", MEASURED["opmix_before"], "ours"),
                          ("mine, after",  MEASURED["opmix_after"],  "ours"),
                          ("V16-RC5",      MEASURED["opmix_after"],  "boatlee")):
    d = blob[seat]
    rows.append({"agent": label, "money": d["money"], "unit-turns": d["unit_turns"],
                 "walking": d["moves"], "idle (PASS)": d["pass"], "WATER": d["water"],
                 "HARVEST": d["harvest"], "PLANT": d["plant"], "CARE": d["care"],
                 "shed trips (PICKUP)": d["pickup"]})
budget = pd.DataFrame(rows).set_index("agent")
display(budget)

b = budget.loc["mine, before"]; v = budget.loc["V16-RC5"]
print(f"\nBefore: {(b['walking'] + b['idle (PASS)']) / b['unit-turns']:.0%} of unit-turns "
      f"went to walking or standing still.")
print(f"V16-RC5: {(v['walking'] + v['idle (PASS)']) / v['unit-turns']:.0%}.")

Two corrections I had to make to my own first reading of this table.

**My agent was not "growing nothing".** I first read the op counts off a truncated
top-9 list, saw no `WATER` or `HARVEST`, and concluded the farm was pure livestock.
It was not: 188 waterings and 198 harvests. The real statement is weaker and more
useful — it grew *a fifth* as much as the opponent (188 vs 1,010 waterings, 45 vs
199 plantings), because twenty-seven animals and their feed filled the 25 starting
tiles before any fruit could be planted.

**Fertiliser is not free money.** I sold 371 units of it, more than V16-RC5's 300, and
banked far less. Fertiliser is the one product with a town drain of exactly zero — no
shop lists it, the town centre skips it — so there is no hole to sell it into and every
unit prices on the shallow glut branch. Spent on a strawberry instead it doubles four
scheduled yields.

## 5. The rebuild, and what each change was worth

Five changes, each kept only if it survived the arena:

| change | why |
|---|---|
| herd cut to 8 cows / 4 sheep / 0 geese | animals cost four unit-turns a day; tiles are not the scarce thing |
| strawberry planted and fertilised | fertilised + watered doubles each of four scheduled yields, 8 units off one tile |
| wheat as filler on every idle tile | $10 a seed for 4–6 units into a 525-unit hole, the cheapest use of a spare tile |
| both cheap quadrants bought by day 10 | 25 tiles cannot hold a herd and its feed, let alone fruit |
| pricing moved to the 1.32.7 hinge | `CARROT`, `TOMATO` and `EGG` changed shape in 1.32.7; my table was 1.32.6 |

That last one deserves a note, because the fix is not the interesting part. In 1.32.7
those three products use a `hinge` below-curve: linear in `x/T`, plus
`8.0 · max(0, x/T − 1)²` past the knee at `T`. The quadratic is identically zero below
the knee — and for two of the three the knee sits *above* what the town eats all season,
so it can never fire at typical scarcity.

In [ ]:
# Season drain per product, from the engine's own shop schedule, against each knee.
from kaggle_environments.envs.kaggriculture.kaggriculture import (
    SHOPS, TOWN_CENTER_PRODUCTS, PRODUCTS,
)
TURNS_PER_DAY, DAYS, SHOP_INTERVAL, CENTER_INTERVAL = 24, 30, 4, 24
MAX_SHOPS, UNLOCK_EVERY = 8, 3

drain = {p: 0.0 for p in PRODUCTS}
for day in range(DAYS):
    n_shops = min(MAX_SHOPS, day // UNLOCK_EVERY)
    ticks = TURNS_PER_DAY // SHOP_INTERVAL
    for products in SHOPS.values():
        copies = n_shops / len(SHOPS)
        mult = 2 if len(products) == 1 else 1
        for item in products:
            drain[item] += copies * mult * ticks
    for item in TOWN_CENTER_PRODUCTS:
        drain[item] += TURNS_PER_DAY // CENTER_INTERVAL

print(f"{'product':<12}{'shape':>8}{'knee T':>8}{'town eats':>11}{'ratio':>8}   fires?")
for item in ("CARROT", "TOMATO", "EGG"):
    p = MARKET_PARAMS[item]
    eaten, T = round(drain[item]), p["T"]
    print(f"{item:<12}{p['below_func']:>8}{T:>8}{eaten:>11}{eaten / T:>7.2f}x"
          f"   {'yes' if eaten > T else 'no, never at median demand'}")

## 6. Result

100 episodes against each configuration — five opponents, ten seeds, both seatings.

In [ ]:
before, after = MEASURED["before"], MEASURED["after"]
rows = []
for name in before["per_opponent"]:
    b, a = before["per_opponent"][name], after["per_opponent"][name]
    rows.append({"opponent": name,
                 "win% before": b["winrate"], "win% after": a["winrate"],
                 "our $ before": b["our_money"], "our $ after": a["our_money"],
                 "their $": a["their_money"]})
res = pd.DataFrame(rows).set_index("opponent")
display(res.style.format({"win% before": "{:.0%}", "win% after": "{:.0%}",
                          "our $ before": "${:,.0f}", "our $ after": "${:,.0f}",
                          "their $": "${:,.0f}"}))
print(f"overall: {before['winrate']:.1%} -> {after['winrate']:.1%} "
      f"over {after['games']} episodes each")

### How many seeds you need before you believe anything

Mean money per seed for the rebuilt agent, averaged over all ten of its games on that
seed. The spread across seeds is wider than every improvement I measured all day.
Three seeds is noise; I had a change that looked like +5 points on three seeds and
was worth nothing on ten.

In [ ]:
import numpy as np
per_seed = MEASURED["after"]["per_seed_mean"]
vals = np.array(list(per_seed.values()))
print(f"per-seed mean money: min ${vals.min():,.0f}  max ${vals.max():,.0f}  "
      f"spread ${vals.max() - vals.min():,.0f}")
print(f"overall mean ${vals.mean():,.0f}, sd across seeds ${vals.std(ddof=1):,.0f}")

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE, RED, AQUA = "#2a78d6", "#eb6834", "#e34948", "#1baf7a"
INK, MUTED, GRID = "#1a1a1a", "#5c5c5c", "#dcdcdc"
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": GRID,
                     "text.color": INK, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9})

names = list(res.index)
x = np.arange(len(names)); w = 0.27
fig, ax = plt.subplots(figsize=(7.4, 3.5))
ax.bar(x - w, res["our $ before"], w, color=RED, label="my agent, before")
ax.bar(x,     res["our $ after"],  w, color=BLUE, label="my agent, after")
ax.bar(x + w, res["their $"],      w, color=ORANGE, label="the opponent")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("mean money banked, 20 episodes each")
ax.yaxis.set_major_formatter(lambda v, p: f"${v/1000:.0f}k")
ax.yaxis.grid(True, color=GRID, lw=0.7); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", ncol=3, fontsize=8)
ax.set_title("The rebuild closes a third of the gap, and no more", loc="left",
             fontsize=10.5, pad=12)
plt.tight_layout(); plt.show()

In [ ]:
cats = ["walking", "idle (PASS)", "WATER", "HARVEST", "PLANT", "CARE", "shed trips (PICKUP)"]
labels = ["mine, before", "mine, after", "V16-RC5"]
colors = [RED, BLUE, ORANGE]

fig, axes = plt.subplots(1, 3, figsize=(8.6, 3.2), sharex=True)
top = budget[cats].values.max() * 1.15
for ax, lab, col in zip(axes, labels, colors):
    heights = [budget.loc[lab, c] for c in cats]
    ax.barh(range(len(cats)), heights, color=col, height=0.62)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats if lab == labels[0] else [])
    ax.invert_yaxis(); ax.set_xlim(0, top)
    ax.xaxis.grid(True, color=GRID, lw=0.7); ax.set_axisbelow(True)
    ax.set_title(lab, fontsize=9, pad=8)
fig.suptitle("Where roughly 7,000 unit-turns of a season go", x=0.01, ha="left",
             fontsize=10.5)
fig.text(0.01, 0.005, "one episode, seed 1000 — turn counts are stable across seeds, "
         "the money on any single seed is not", fontsize=7.5, color=MUTED)
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.0))
seeds = list(per_seed.keys())
ax.bar(range(len(seeds)), list(per_seed.values()), color=BLUE, width=0.62)
ax.axhline(vals.mean(), color=ORANGE, lw=2, zorder=3)
ax.annotate(f"mean ${vals.mean():,.0f}", (-0.4, vals.mean()),
            xytext=(0, 7), textcoords="offset points", ha="left",
            color=ORANGE, fontsize=8.5, fontweight="bold")
ax.set_xticks(range(len(seeds))); ax.set_xticklabels(seeds, fontsize=8)
ax.set_xlabel("environment seed")
ax.set_ylabel("mean money over that seed's 10 games")
ax.yaxis.set_major_formatter(lambda v, p: f"${v/1000:.0f}k")
ax.yaxis.grid(True, color=GRID, lw=0.7); ax.set_axisbelow(True)
ax.set_title("Seed noise is bigger than any change I made", loc="left",
             fontsize=10.5, pad=12)
plt.tight_layout(); plt.show()

## 7. The agent

Fork it, change the constants in `P`, and run it through `arena()` above against agents
you extract yourself. Everything it prices from is recomputed from the installed
engine, so it does not go stale the next time the market parameters move.

What I would try next, in the order I think it pays: the walking figure is still 63% of
unit-turns against V16-RC5's 43%, and none of my parameter sweeps touched it — the
routing, not the economics, is what is left. Past that, the top of this board is doing
offline sequence search, and no amount of heuristic tuning reaches it.

In [ ]:
import tarfile
with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py")

env = make("kaggriculture", configuration={"seed": 1014}, debug=False)
env.run(["main.py", "starter"])
print(f"runs clean: ${env.steps[-1][0]['reward']:,.0f} on seed 1014")
print("submission.tar.gz written")

Corrections welcome, particularly on the seed count — ten may still be too few, and if
someone has run a few hundred I would like to know how much of the 3.0% → 8.0% survives.